# LUMI: Shoplifting detection, fine-tuning VideoMAE on DCSASS
**Owner:** Shohruh · **Runs on:** Kaggle (GPU P100 or T4, Internet ON)

Before you run:
1. *Add Input* → search **DCSASS Dataset** (`mateohervas/dcsass-dataset`) → Add.
2. *Settings* → Accelerator **GPU T4** (single T4 is enough; P100 also works) · Internet **On**.
3. Optional: *Add-ons → Secrets* → add `HF_TOKEN` (write token) to push the model to Hugging Face.
4. Click **Save Version → Save & Run All** so it trains in the background.

Pipeline: find clips + labels → split **by source video** → sample 16 frames → fine-tune → evaluate → save.

In [ ]:
# Kaggle already has torch/torchvision/transformers/opencv/sklearn installed.
import os, glob, json, random, math, time
from pathlib import Path
import numpy as np, pandas as pd, cv2

CFG = dict(
    DATA_ROOT="/kaggle/input",            # searched recursively for DCSASS
    CATEGORY="Shoplifting",               # DCSASS category to use
    MODEL_ID="MCG-NJU/videomae-base-finetuned-kinetics",
    NUM_FRAMES=16, IMG_SIZE=224,
    BATCH_SIZE=4, GRAD_ACCUM=4,           # effective batch 16
    EPOCHS=10, LR=5e-5, WEIGHT_DECAY=0.05, WARMUP_RATIO=0.1,
    NUM_WORKERS=2, SEED=42,
    SPLIT=(0.70, 0.15, 0.15),             # by source video
    OUT_DIR="/kaggle/working/videomae-shoplifting",
    HF_REPO=None,                         # e.g. "your-hf-name/lumi-videomae-shoplifting" to push
    DEBUG_LIMIT=None,                     # e.g. 40 to do a quick smoke run first
)
def seed_all(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError: pass
seed_all(CFG["SEED"])

## 1. Build the manifest (one row per clip)
DCSASS layout: `.../Shoplifting/Shoplifting001_x264.mp4/Shoplifting001_x264_0.mp4` (a *folder* per source video holding its short clips),
and labels in `.../Labels/Shoplifting.csv` with rows like `Shoplifting001_x264_0, Shoplifting, 0|1`.

In [ ]:
def find_label_csv(root, category):
    hits = [p for p in glob.glob(f"{root}/**/{category}.csv", recursive=True)]
    if not hits: raise FileNotFoundError(f"No {category}.csv under {root}. Did you add the DCSASS dataset as input?")
    return hits[0]

def read_labels(csv_path):
    df = pd.read_csv(csv_path, header=None)
    # Some copies have a header row; drop rows where last column isn't 0/1.
    df = df[df.iloc[:, -1].astype(str).str.strip().isin(["0", "1", "0.0", "1.0"])]
    out = pd.DataFrame({"clip": df.iloc[:, 0].astype(str).str.strip(),
                        "label": df.iloc[:, -1].astype(float).astype(int)})
    out["clip"] = out["clip"].str.replace(r"\.mp4$", "", regex=True)
    return out

def build_manifest(root, category):
    labels = read_labels(find_label_csv(root, category))
    vids = {Path(p).stem: p for p in glob.glob(f"{root}/**/{category}*.mp4", recursive=True) if os.path.isfile(p)}
    labels["path"] = labels["clip"].map(vids)
    missing = labels["path"].isna().sum()
    labels = labels.dropna(subset=["path"]).reset_index(drop=True)
    # Shoplifting001_x264_7 -> Shoplifting001_x264
    labels["source_video"] = labels["clip"].str.rsplit("_", n=1).str[0]
    print(f"clips with video: {len(labels)}  (labels without a file: {missing})")
    return labels

manifest = build_manifest(CFG["DATA_ROOT"], CFG["CATEGORY"])
print(manifest["label"].value_counts().rename({0: "normal", 1: "shoplifting"}))
print("source videos:", manifest["source_video"].nunique())
manifest.head()

## 2. Split by source video (prevents leakage)
Clips cut from the same video look almost identical, so a random clip split would give fake 95%+ scores.
We split whole source videos, stratified by "does this video contain any shoplifting clip".

In [ ]:
def split_by_video(df, fracs, seed):
    rng = np.random.default_rng(seed)
    has_theft = df.groupby("source_video")["label"].max()
    split = {}
    for flag in (0, 1):
        vids = has_theft[has_theft == flag].index.to_numpy().copy(); rng.shuffle(vids)
        n = len(vids); n_tr = int(round(n * fracs[0])); n_va = int(round(n * fracs[1]))
        if flag == 1 and n >= 3:   # make sure val and test each get theft videos
            n_va = max(n_va, 1); n_tr = min(n_tr, n - n_va - 1)
        for i, v in enumerate(vids):
            split[v] = "train" if i < n_tr else ("val" if i < n_tr + n_va else "test")
    df = df.copy(); df["split"] = df["source_video"].map(split)
    return df

manifest = split_by_video(manifest, CFG["SPLIT"], CFG["SEED"])
if CFG["DEBUG_LIMIT"]:
    manifest = manifest.groupby("split", group_keys=False).apply(lambda g: g.sample(min(len(g), CFG["DEBUG_LIMIT"]), random_state=0))
assert not (set(manifest[manifest.split=="train"].source_video) & set(manifest[manifest.split=="test"].source_video)), "leakage!"
print(pd.crosstab(manifest["split"], manifest["label"], margins=True))
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
manifest.to_csv(f"{CFG['OUT_DIR']}/manifest.csv", index=False)

## 3. Read 16 evenly spaced frames per clip

In [ ]:
def read_frames(path, num_frames):
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        raise RuntimeError(f"could not read {path}")
    idx = np.linspace(0, len(frames) - 1, num_frames).round().astype(int)
    return np.stack([frames[i] for i in idx])   # (T, H, W, 3) uint8

sample = read_frames(manifest.iloc[0]["path"], CFG["NUM_FRAMES"])
print("frames:", sample.shape)

## 4. Dataset + augmentation
torchvision v2 transforms on a (T,C,H,W) tensor apply the **same** random crop/flip/colour to every frame, keeping motion consistent.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from transformers import VideoMAEForVideoClassification

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet stats used by VideoMAE
S = CFG["IMG_SIZE"]

train_tf = v2.Compose([
    v2.RandomResizedCrop(S, scale=(0.6, 1.0), ratio=(0.75, 1.33), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])
eval_tf = v2.Compose([
    v2.Resize(S, antialias=True), v2.CenterCrop(S),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

class ClipDataset(Dataset):
    def __init__(self, df, tf): self.df, self.tf = df.reset_index(drop=True), tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = torch.from_numpy(read_frames(r["path"], CFG["NUM_FRAMES"])).permute(0, 3, 1, 2)  # T,C,H,W uint8
        return self.tf(x), torch.tensor(r["label"], dtype=torch.long)

dfs = {s: manifest[manifest.split == s] for s in ("train", "val", "test")}
loaders = {s: DataLoader(ClipDataset(d, train_tf if s == "train" else eval_tf),
                         batch_size=CFG["BATCH_SIZE"], shuffle=(s == "train"),
                         num_workers=CFG["NUM_WORKERS"], pin_memory=True, drop_last=(s == "train"))
           for s, d in dfs.items()}
xb, yb = next(iter(loaders["train"])); print("batch:", xb.shape, yb)   # (B,16,3,224,224)

## 5. Model: Kinetics-pretrained VideoMAE with a new 2-class head

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
id2label = {0: "normal", 1: "shoplifting"}
model = VideoMAEForVideoClassification.from_pretrained(
    CFG["MODEL_ID"], num_labels=2, id2label=id2label, label2id={v: k for k, v in id2label.items()},
    ignore_mismatched_sizes=True).to(device)

# Class weights: shoplifting is the minority class (~1:5)
counts = dfs["train"]["label"].value_counts().reindex([0, 1], fill_value=1).to_numpy()
class_w = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32, device=device)
print("class weights:", class_w.tolist())
loss_fn = torch.nn.CrossEntropyLoss(weight=class_w)

opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
steps = math.ceil(len(loaders["train"]) / CFG["GRAD_ACCUM"]) * CFG["EPOCHS"]
warm = max(1, int(CFG["WARMUP_RATIO"] * steps))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                          else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

## 6. Train (keep best epoch by validation PR-AUC)

In [ ]:
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_fscore_support,
                             confusion_matrix, precision_recall_curve)

@torch.no_grad()
def predict(loader):
    model.eval(); probs, ys = [], []
    for x, y in loader:
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(pixel_values=x.to(device, non_blocking=True)).logits
        probs.append(torch.softmax(logits.float(), -1)[:, 1].cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(probs).numpy()

def safe_auc(fn, y, p):
    return float(fn(y, p)) if len(np.unique(y)) == 2 else float("nan")

best_ap, history = -1, []
for epoch in range(1, CFG["EPOCHS"] + 1):
    model.train(); t0, run_loss = time.time(), 0.0
    opt.zero_grad(set_to_none=True)
    for step, (x, y) in enumerate(loaders["train"], 1):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            loss = loss_fn(model(pixel_values=x.to(device, non_blocking=True)).logits.float(), y.to(device)) / CFG["GRAD_ACCUM"]
        scaler.scale(loss).backward(); run_loss += loss.item() * CFG["GRAD_ACCUM"]
        if step % CFG["GRAD_ACCUM"] == 0 or step == len(loaders["train"]):
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
    yv, pv = predict(loaders["val"])
    ap, auc = safe_auc(average_precision_score, yv, pv), safe_auc(roc_auc_score, yv, pv)
    history.append(dict(epoch=epoch, train_loss=run_loss / len(loaders["train"]), val_ap=ap, val_auc=auc))
    print(f"epoch {epoch:2d}  loss {history[-1]['train_loss']:.4f}  val PR-AUC {ap:.3f}  ROC-AUC {auc:.3f}  ({time.time()-t0:.0f}s)")
    if not np.isnan(ap) and ap > best_ap:
        best_ap = ap; model.save_pretrained(CFG["OUT_DIR"])
        print("   ↳ saved best")
if best_ap < 0:   # val had a single class; just keep the last epoch
    model.save_pretrained(CFG["OUT_DIR"])
pd.DataFrame(history).to_csv(f"{CFG['OUT_DIR']}/history.csv", index=False)

## 7. Evaluate best model: pick threshold on **val**, report on **test**

In [ ]:
model = VideoMAEForVideoClassification.from_pretrained(CFG["OUT_DIR"]).to(device)
yv, pv = predict(loaders["val"]); yt, pt = predict(loaders["test"])

prec, rec, thr = precision_recall_curve(yv, pv)
f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
threshold = float(thr[np.argmax(f1[:-1])]) if len(thr) else 0.5
print(f"threshold chosen on val (max F1): {threshold:.3f}")

pred = (pt >= threshold).astype(int)
p, r, f, _ = precision_recall_fscore_support(yt, pred, average="binary", zero_division=0)
cm = confusion_matrix(yt, pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
metrics = dict(threshold=threshold, test_precision=float(p), test_recall=float(r), test_f1=float(f),
               test_roc_auc=safe_auc(roc_auc_score, yt, pt), test_pr_auc=safe_auc(average_precision_score, yt, pt),
               false_alarm_rate=float(fp / max(fp + tn, 1)),   # share of normal clips that would alert
               confusion_matrix={"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
               n_test_clips=int(len(yt)), n_test_videos=int(dfs["test"].source_video.nunique()))
print(json.dumps(metrics, indent=2))
json.dump(metrics, open(f"{CFG['OUT_DIR']}/metrics.json", "w"), indent=2)
json.dump({"threshold": threshold, "num_frames": CFG["NUM_FRAMES"], "img_size": S, "mean": MEAN, "std": STD, "base_model": CFG["MODEL_ID"]},
          open(f"{CFG['OUT_DIR']}/inference_config.json", "w"), indent=2)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
h = pd.DataFrame(history)
ax[0].plot(h.epoch, h.train_loss, label="train loss"); ax[0].plot(h.epoch, h.val_ap, label="val PR-AUC"); ax[0].legend(); ax[0].set_xlabel("epoch")
ax[1].imshow(cm, cmap="Blues"); ax[1].set_xticks([0, 1], ["normal", "shoplifting"]); ax[1].set_yticks([0, 1], ["normal", "shoplifting"])
for (i, j), v in np.ndenumerate(cm): ax[1].text(j, i, v, ha="center", va="center")
ax[1].set_xlabel("predicted"); ax[1].set_ylabel("true"); ax[1].set_title("test confusion matrix")
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/results.png", dpi=120); plt.show()

## 8. Quick inference check + optional push to Hugging Face
The backend will load the model the same way: `from_pretrained(<folder or HF repo>)` + `inference_config.json` for the threshold.

In [ ]:
def classify_clip(path):
    x = eval_tf(torch.from_numpy(read_frames(path, CFG["NUM_FRAMES"])).permute(0, 3, 1, 2)).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(pixel_values=x).logits.float(), -1)[0, 1].item()
    return prob, prob >= threshold

for _, r in dfs["test"].sample(min(5, len(dfs["test"])), random_state=1).iterrows():
    prob, alert = classify_clip(r["path"])
    print(f"{r['clip']:<32} true={r['label']}  p(shoplifting)={prob:.2f}  alert={alert}")

if CFG["HF_REPO"]:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login, HfApi
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    HfApi().create_repo(CFG["HF_REPO"], private=True, exist_ok=True)
    HfApi().upload_folder(folder_path=CFG["OUT_DIR"], repo_id=CFG["HF_REPO"])
    print("pushed to", CFG["HF_REPO"])
print("Done. Files in", CFG["OUT_DIR"], os.listdir(CFG["OUT_DIR"]))